# Week 3 Linear Regression & Regularization

**Datasets:** Taobao User Behavior · Olist Brazilian E-Commerce · UCL Online Retail II

## Deep-Dive Topics 
1. **Multicollinearity & regularization (Olist)** — total_price and payment_value are nearly collinear. We reuse the VIF / OLS / Lasso / Ridge comparison from the lecture notebook on real order data.
2. **Forward vs. backward selection (UCL)** — We reuse the lecture stepwise-selection code on UCL transaction data and compare which features enter or leave first.

## Data loading and clearning 

Load and prepare modeling dfs for all three datasets.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
DATA_DIR = Path(".")

orders = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_orders_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_reviews_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_payments_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_products_dataset.csv")

pay = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    n_payments=("payment_sequential", "max"),
).reset_index()
itm = items.groupby("order_id").agg(
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    n_items=("order_item_id", "count"),
    avg_item_price=("price", "mean"),
).reset_index()
first_item = items.sort_values("order_item_id").groupby("order_id").first().reset_index()
prod_agg = first_item.merge(products, on="product_id", how="left").groupby("order_id").agg(
    product_weight_g=("product_weight_g", "mean"),
    product_length_cm=("product_length_cm", "mean"),
    product_height_cm=("product_height_cm", "mean"),
    product_width_cm=("product_width_cm", "mean"),
    product_photos_qty=("product_photos_qty", "mean"),
).reset_index()

olist = (
    reviews.merge(orders, on="order_id")
    .merge(itm, on="order_id")
    .merge(pay, on="order_id")
    .merge(prod_agg, on="order_id", how="left")
)
olist["order_purchase_timestamp"] = pd.to_datetime(olist["order_purchase_timestamp"])
olist["order_delivered_customer_date"] = pd.to_datetime(olist["order_delivered_customer_date"])
olist["delivery_days"] = (
    olist["order_delivered_customer_date"] - olist["order_purchase_timestamp"]
).dt.days

olist_features = [
    "total_price", "total_freight", "n_items", "avg_item_price",
    "payment_value", "payment_installments", "n_payments", "delivery_days",
    "product_weight_g", "product_length_cm", "product_height_cm",
    "product_width_cm", "product_photos_qty",
]
olist_df = olist.dropna(subset=["review_score", "delivery_days"]).copy()
olist_df[olist_features] = olist_df[olist_features].fillna(olist_df[olist_features].median())

ucl = pd.read_excel(DATA_DIR / "ucl_dataset.xlsx")
ucl = ucl.rename(columns={"Customer ID": "CustomerID", "Price": "UnitPrice"})
ucl["InvoiceDate"] = pd.to_datetime(ucl["InvoiceDate"])
ucl["line_total"] = ucl["Quantity"] * ucl["UnitPrice"]
ucl = ucl[(ucl["Quantity"] > 0) & (ucl["UnitPrice"] > 0)].dropna(subset=["CustomerID"])
ucl["month"] = ucl["InvoiceDate"].dt.month
ucl["dayofweek"] = ucl["InvoiceDate"].dt.dayofweek
ucl["hour"] = ucl["InvoiceDate"].dt.hour
ucl["is_weekend"] = (ucl["dayofweek"] >= 5).astype(int)
ucl_features = ["UnitPrice", "month", "dayofweek", "hour", "is_weekend", "CustomerID"]
ucl_df = ucl.sample(n=min(8000, len(ucl)), random_state=42)

TAOBAO_PATH = DATA_DIR / "TaobaoUserBehavior.csv"
SAMPLE_SIZE = 400_000
chunks, rows_seen = [], 0
for chunk in pd.read_csv(
    TAOBAO_PATH, header=None,
    names=["user_id", "item_id", "category_id", "behavior", "timestamp"],
    chunksize=200_000,
):
    rows_seen += len(chunk)
    chunks.append(chunk.sample(frac=min(1.0, SAMPLE_SIZE / rows_seen), random_state=42))
    if sum(len(c) for c in chunks) >= SAMPLE_SIZE:
        break

taobao = pd.concat(chunks, ignore_index=True).head(SAMPLE_SIZE)
for beh, col in [("pv", "is_pv"), ("buy", "is_buy"), ("cart", "is_cart"), ("fav", "is_fav")]:
    taobao[col] = (taobao["behavior"] == beh).astype(int)
taobao_df = taobao.groupby(["user_id", "category_id"]).agg(
    n_pv=("is_pv", "sum"), n_buy=("is_buy", "sum"),
    n_cart=("is_cart", "sum"), n_fav=("is_fav", "sum"),
).reset_index()
taobao_df = taobao_df[taobao_df["n_pv"] > 0]
taobao_features = ["n_pv", "n_cart", "n_fav"]

print(f"Olist rows: {len(olist_df):,}")
print(f"UCL sample rows: {len(ucl_df):,}")
print(f"Taobao user-category rows: {len(taobao_df):,}")


Olist rows: 96,358
UCL sample rows: 8,000
Taobao user-category rows: 91,396


# Linear regression fit to all features

Predict Olist review_score from order-level features.

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

feature_names = olist_features
df = olist_df[feature_names + ["review_score"]].copy()

X_all = df[feature_names]
y = df["review_score"]

model = sm.OLS(y, X_all).fit()

highlight = {"total_price", "payment_value"}

print("Estimated coefficients:")
for name, coef in model.params.items():
    if name in highlight:
        print(f"*** {name}: {coef: .6f}  <-- collinear pair")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nModel summary:")
print(model.summary())


Estimated coefficients:
*** total_price:  0.006513  <-- collinear pair
    total_freight:  0.007444
    n_items:  0.885467
    avg_item_price:  0.005172
*** payment_value: -0.011011  <-- collinear pair
    payment_installments:  0.045672
    n_payments:  1.420015
    delivery_days: -0.019921
    product_weight_g: -0.000107
    product_length_cm:  0.016783
    product_height_cm:  0.022815
    product_width_cm:  0.027046
    product_photos_qty:  0.155401

Model summary:
                                 OLS Regression Results                                
Dep. Variable:           review_score   R-squared (uncentered):                   0.873
Model:                            OLS   Adj. R-squared (uncentered):              0.873
Method:                 Least Squares   F-statistic:                          5.105e+04
Date:                Sun, 07 Jun 2026   Prob (F-statistic):                        0.00
Time:                        20:33:48   Log-Likelihood:                     -1.7887e+05

# Lasso regression

 Predict Taobao n_buy from engagement counts (n_pv, n_cart, n_fav).

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

feature_names = taobao_features
df = taobao_df[feature_names + ["n_buy"]].copy()

X_all = df[feature_names].values
y = df["n_buy"].values

scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X_all)

lambda_param = 0.05

lasso = Lasso(alpha=lambda_param, fit_intercept=False, max_iter=10000)
lasso.fit(X_scaled, y)

print(f"Lasso coefficients with lambda = {lambda_param}:")
for name, coef in zip(feature_names, lasso.coef_):
    if abs(coef) > 1e-8:
        print(f"*** {name}: {coef: .6f}  <-- nonzero")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nNumber of nonzero coefficients:", np.sum(np.abs(lasso.coef_) > 1e-8))


Lasso coefficients with lambda = 0.05:
*** n_pv:  0.013930  <-- nonzero
    n_cart:  0.000000
    n_fav:  0.000000

Number of nonzero coefficients: 1


# Ridge regression

Predict Olist review_score with Ridge when price and payment features overlap.

In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

feature_names = olist_features
df = olist_df[feature_names + ["review_score"]].copy()

X_all = df[feature_names].values
y = df["review_score"].values

scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X_all)

lambda_param = 5.0

ridge = Ridge(alpha=lambda_param, fit_intercept=False)
ridge.fit(X_scaled, y)

print(f"Ridge coefficients with lambda = {lambda_param}:")
for name, coef in zip(feature_names, ridge.coef_):
    if name in {"total_price", "payment_value"}:
        print(f"*** {name}: {coef: .6f}  <-- collinear monetary features")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nNote: ridge usually keeps all coefficients nonzero,")
print("but shrinks the coefficients of less important features toward 0.")


Ridge coefficients with lambda = 5.0:
*** total_price:  0.032531  <-- collinear monetary features
    total_freight:  0.072749
    n_items: -0.179263
    avg_item_price:  0.071627
*** payment_value: -0.108343  <-- collinear monetary features
    payment_installments: -0.011297
    n_payments: -0.004492
    delivery_days: -0.440270
    product_weight_g: -0.041287
    product_length_cm:  0.007298
    product_height_cm:  0.008870
    product_width_cm:  0.001345
    product_photos_qty: -0.003219

Note: ridge usually keeps all coefficients nonzero,
but shrinks the coefficients of less important features toward 0.


X1 = 1000000, X2 = 1
Equation: X1 + 1000000 * X2
Lasso or ridge don't work on this
standard scaler
X1* = X1 / 1000000 = 1, X2 = 1
Equation: 1000000 * X1* + 1000000 * X2

Same  applies to Olist: total_price and payment_value are on very different scales before standardization.

# Ridge regression with various values of lambda


In [5]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

feature_names = olist_features
df = olist_df[feature_names + ["review_score"]].copy()

X_all = df[feature_names].values
y = df["review_score"].values

pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("ridge", Ridge(fit_intercept=False))
])

lambda_grid = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30]

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid={"ridge__alpha": lambda_grid},
    scoring="neg_mean_squared_error",
    cv=cv,
    return_train_score=True
)

grid.fit(X_all, y)

print("Best lambda (alpha) found by GridSearchCV:")
print(grid.best_params_["ridge__alpha"])
print(f"Best CV score (negative MSE): {grid.best_score_:.6f}")

best_model = grid.best_estimator_
best_ridge = best_model.named_steps["ridge"]
best_lambda = best_model.named_steps["ridge"].alpha

print(f"\nRidge coefficients with best lambda = {best_lambda}:")
for name, coef in zip(feature_names, best_ridge.coef_):
    if name in {"total_price", "payment_value"}:
        print(f"*** {name}: {coef: .6f}  <-- collinear monetary features")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nNote: ridge usually keeps all coefficients nonzero,")
print("but shrinks the coefficients of less important features toward 0.")

print("\nGrid search results:")
for alpha, mean_score in zip(grid.cv_results_["param_ridge__alpha"], grid.cv_results_["mean_test_score"]):
    print(f"lambda = {alpha}: mean CV neg MSE = {mean_score:.6f}")


Best lambda (alpha) found by GridSearchCV:
30
Best CV score (negative MSE): -18.706889

Ridge coefficients with best lambda = 30:
*** total_price: -0.025387  <-- collinear monetary features
    total_freight:  0.066597
    n_items: -0.179213
    avg_item_price:  0.071390
*** payment_value: -0.047451  <-- collinear monetary features
    payment_installments: -0.011315
    n_payments: -0.004491
    delivery_days: -0.440139
    product_weight_g: -0.041218
    product_length_cm:  0.007284
    product_height_cm:  0.008864
    product_width_cm:  0.001345
    product_photos_qty: -0.003210

Note: ridge usually keeps all coefficients nonzero,
but shrinks the coefficients of less important features toward 0.

Grid search results:
lambda = 0.01: mean CV neg MSE = -18.706916
lambda = 0.03: mean CV neg MSE = -18.706915
lambda = 0.1: mean CV neg MSE = -18.706913
lambda = 0.3: mean CV neg MSE = -18.706909
lambda = 1.0: mean CV neg MSE = -18.706903
lambda = 3.0: mean CV neg MSE = -18.706898
lambda = 1

# Elastic Net regression


Olist review_score with both L1 and L2 penalties.

In [6]:
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler

feature_names = olist_features
df = olist_df[feature_names + ["review_score"]].copy()

X_all = df[feature_names].values
y = df["review_score"].values

scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X_all)

lambda_param = 0.05
l1_ratio = 0.5

enet = ElasticNet(alpha=lambda_param, l1_ratio=l1_ratio, fit_intercept=False, max_iter=10000)
enet.fit(X_scaled, y)

print(f"Elastic Net coefficients (lambda = {lambda_param}, l1_ratio = {l1_ratio}):")
for name, coef in zip(feature_names, enet.coef_):
    if abs(coef) > 1e-8:
        print(f"*** {name}: {coef: .6f}  <-- nonzero")
    else:
        print(f"    {name}: {coef: .6f}")

print("\nNumber of nonzero coefficients:", np.sum(np.abs(enet.coef_) > 1e-8))


Elastic Net coefficients (lambda = 0.05, l1_ratio = 0.5):
    total_price: -0.000000
    total_freight:  0.000000
*** n_items: -0.138048  <-- nonzero
    avg_item_price:  0.000000
    payment_value: -0.000000
    payment_installments: -0.000000
    n_payments: -0.000000
*** delivery_days: -0.396551  <-- nonzero
    product_weight_g: -0.000000
    product_length_cm:  0.000000
    product_height_cm:  0.000000
    product_width_cm:  0.000000
    product_photos_qty: -0.000000

Number of nonzero coefficients: 2


# Ordinary, lasso, and ridge regression with collinearity and VIF

### Deep Dive Topic 1: Multicollinearity & regularization on Olist

On Olist, total_price and payment_value measure almost the same thing (same VIF problem as the lecture's collinear x1, x2).

In [7]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler

df = olist_df[["total_price", "payment_value", "review_score"]].dropna().copy()

print("Collinear pair: total_price, payment_value")
print("Target: review_score")
print("Data sample:")
print(df.head())
print()

X = df[["total_price", "payment_value"]]

vif_df = pd.DataFrame()
vif_df["feature"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print("VIF values:")
print(vif_df)
print()

ols_model = sm.OLS(df["review_score"], X).fit()

print("Regular regression (OLS) coefficients:")
for name, coef in ols_model.params.items():
    print(f"{name}: {coef: .6f}")
print()

scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X.values)

lasso_lambda = 0.02
lasso = Lasso(alpha=lasso_lambda, fit_intercept=False, max_iter=10000)
lasso.fit(X_scaled, df["review_score"].values)

print(f"Lasso coefficients (lambda = {lasso_lambda}):")
for name, coef in zip(X.columns, lasso.coef_):
    print(f"{name}: {coef: .6f}")
print()

ridge_lambda = 0.5
ridge = Ridge(alpha=ridge_lambda, fit_intercept=False)
ridge.fit(X_scaled, df["review_score"].values)

print(f"Ridge coefficients (lambda = {ridge_lambda}):")
for name, coef in zip(X.columns, ridge.coef_):
    print(f"{name}: {coef: .6f}")
print()


Collinear pair: total_price, payment_value
Target: review_score
Data sample:
   total_price  payment_value  review_score
0       370.00         397.26             4
1        79.79          88.09             5
2       149.00         194.12             5
3       179.99         222.84             5
4      1199.00        1333.25             5

VIF values:
         feature         VIF
0    total_price  123.548329
1  payment_value  123.548329

Regular regression (OLS) coefficients:
total_price: -0.078161
payment_value:  0.080744

Lasso coefficients (lambda = 0.02):
total_price: -0.000000
payment_value: -0.034115

Ridge coefficients (lambda = 0.5):
total_price:  1.128447
payment_value: -1.177921



total_price and payment_value are nearly interchangeable in Olist. so OLS assigns unstable opposite-signed weights. Lasso may zero one out, and Ridge shrinks both toward similar values.

# Explanation of regular, lasso, and ridge regression loss terms

In [8]:
def print_loss_functions():
    print("1) Regular linear regression (two features: x1, x2, no intercept)")
    print("   The usual loss is the sum of squared errors (SSE):")
    print("   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2")
    print("   In words: choose the two coefficients to minimize squared prediction error.")
    print()

    print("2) Lasso regression (no intercept)")
    print("   Lasso adds an L1 penalty with lambda:")
    print("   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2")
    print("                    + lambda * (|beta1| + |beta2|)")
    print("   In words: minimize squared prediction error, but also penalize the absolute size")
    print("   of the coefficients. This can force some coefficients exactly to zero.")
    print()

    print("3) Ridge regression (no intercept)")
    print("   Ridge adds an L2 penalty with lambda:")
    print("   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2")
    print("                    + lambda * (beta1^2 + beta2^2)")
    print("   In words: minimize squared prediction error, but also penalize large coefficients.")
    print("   This shrinks coefficients toward zero, but usually does not make them exactly zero.")
    print()

    print("Notes:")
    print("- lambda controls the strength of regularization.")
    print("- When lambda = 0, lasso and ridge both reduce to ordinary least squares.")
    print("- Since there is no intercept here, all three formulas use only beta1 and beta2.")

print_loss_functions()


1) Regular linear regression (two features: x1, x2, no intercept)
   The usual loss is the sum of squared errors (SSE):
   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2
   In words: choose the two coefficients to minimize squared prediction error.

2) Lasso regression (no intercept)
   Lasso adds an L1 penalty with lambda:
   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2
                    + lambda * (|beta1| + |beta2|)
   In words: minimize squared prediction error, but also penalize the absolute size
   of the coefficients. This can force some coefficients exactly to zero.

3) Ridge regression (no intercept)
   Ridge adds an L2 penalty with lambda:
   L(beta1, beta2) = sum_i [y_i - (beta1*x1_i + beta2*x2_i)]^2
                    + lambda * (beta1^2 + beta2^2)
   In words: minimize squared prediction error, but also penalize large coefficients.
   This shrinks coefficients toward zero, but usually does not make them exactly zero.

Notes:
- lambda controls

# Forward selection

### Deep Dive Topic 2: Stepwise selection on UCL

Predict UCL line_total from calendar and customer features.

In [9]:
X = ucl_df[ucl_features].copy()
y = ucl_df["line_total"].copy()

def forward_selection(X, y, max_features=10):
    remaining = list(X.columns)
    selected = []
    history = []

    for step in range(max_features):
        best_feature = None
        best_score = np.inf

        for feature in remaining:
            candidate_features = selected + [feature]
            model = LinearRegression()
            scores = cross_val_score(
                model,
                X[candidate_features],
                y,
                cv=5,
                scoring="neg_mean_squared_error"
            )
            mse = -scores.mean()

            if mse < best_score:
                best_score = mse
                best_feature = feature

        selected.append(best_feature)
        remaining.remove(best_feature)
        history.append((step + 1, best_feature, best_score))

    return selected, history

selected_features, history = forward_selection(X, y, max_features=len(ucl_features))

print("Forward selection order:")
for step, feature, score in history:
    print(f"Step {step}: add {feature:>10s} | CV MSE = {score:.4f}")

print("\nFinal selected order:")
print(selected_features)

print("\nCV MSE by number of selected features:")
for k in range(1, len(ucl_features) + 1):
    feats = selected_features[:k]
    model = LinearRegression()
    scores = cross_val_score(
        model,
        X[feats],
        y,
        cv=5,
        scoring="neg_mean_squared_error"
    )
    mse = -scores.mean()
    print(f"{k} features: {mse:.4f}")


NameError: name 'cross_val_score' is not defined

# Backward selection

In [ ]:
X = ucl_df[ucl_features].copy()
y = ucl_df["line_total"].copy()

def cv_mse(X, y, features):
    model = LinearRegression()
    scores = cross_val_score(
        model,
        X[features],
        y,
        cv=5,
        scoring="neg_mean_squared_error"
    )
    return -scores.mean()

def backward_selection(X, y, min_features=1):
    selected = list(X.columns)
    history = []

    while len(selected) > min_features:
        best_score = np.inf
        best_candidate = None
        feature_removed = None

        for feature in selected:
            candidate = [f for f in selected if f != feature]
            mse = cv_mse(X, y, candidate)

            if mse < best_score:
                best_score = mse
                best_candidate = candidate
                feature_removed = feature

        selected = best_candidate
        history.append((feature_removed, selected.copy(), best_score))

    return selected, history

selected_features, history = backward_selection(X, y, min_features=1)

print("Backward selection path:")
print(f"Start with all {len(ucl_features)} features | CV MSE = {cv_mse(X, y, list(X.columns)):.4f}")
for removed_feature, remaining_features, score in history:
    print(f"Remove {removed_feature:>10s} -> keep {remaining_features} | CV MSE = {score:.4f}")


Backward selection path:
Start with all 6 features | CV MSE = 4915.0069
Remove CustomerID -> keep ['UnitPrice', 'month', 'dayofweek', 'hour', 'is_weekend'] | CV MSE = 4913.0024
Remove      month -> keep ['UnitPrice', 'dayofweek', 'hour', 'is_weekend'] | CV MSE = 4911.7904
Remove  dayofweek -> keep ['UnitPrice', 'hour', 'is_weekend'] | CV MSE = 4913.5328
Remove       hour -> keep ['UnitPrice', 'is_weekend'] | CV MSE = 4926.0821
Remove is_weekend -> keep ['UnitPrice'] | CV MSE = 4941.9260


# PCA and PCR where one column varies less 
 Taobao engagement features (n_pv, n_cart, n_fav) predicting n_buy.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

print("Taobao engagement features: n_pv, n_cart, n_fav")
print("Target: n_buy\n")

X1 = taobao_df["n_pv"].values
X2 = taobao_df["n_cart"].values
X3 = taobao_df["n_fav"].values

X = np.column_stack([X1, X2, X3])
y = taobao_df["n_buy"].values

print("Empirical standard deviations:")
print("  std(n_pv)  =", X1.std(ddof=1))
print("  std(n_cart)=", X2.std(ddof=1))
print("  std(n_fav) =", X3.std(ddof=1))
print()

corr = np.corrcoef(X, rowvar=False)
print("Empirical correlation matrix:")
print(corr)
print()

print("Running PCA on X (no standardization, so variance dominates)...")
pca = PCA(n_components=3)
pca.fit(X)

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)
print()

print("Principal component vectors (rows):")
for i, vec in enumerate(pca.components_, start=1):
    print(f"  PC{i}: {vec}")
print()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

def run_pcr(k):
    print(f"Running PCR with k = {k} principal components...")
    model = Pipeline([
        ("pca", PCA(n_components=k)),
        ("lr", LinearRegression())
    ])
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

    print("  Train R^2 :", train_r2)
    print("  Test  R^2 :", test_r2)
    print("  Train RMSE:", train_rmse)
    print("  Test  RMSE:", test_rmse)

    pca_step = model.named_steps["pca"]
    lr_step = model.named_steps["lr"]

    print("  PCA components used by PCR:")
    for i, vec in enumerate(pca_step.components_, start=1):
        print(f"    PC{i}: {vec}")

    print("  Regression coefficients in PC space:")
    print("   ", lr_step.coef_)
    print("  Intercept:")
    print("   ", lr_step.intercept_)
    print()

    return model

print("Now running PCR with different numbers of PCs on Taobao...\n")

model1 = run_pcr(1)
model2 = run_pcr(2)
model3 = run_pcr(3)

print("Running ordinary linear regression on the original columns for comparison...")
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_test_lr = lr.predict(X_test)
print("  Test R^2 :", r2_score(y_test, y_pred_test_lr))
print("  Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test_lr)))
print("  Coefficients on original columns:")
print("   ", lr.coef_)
print("  Intercept:")
print("   ", lr.intercept_)
print()


Taobao engagement features: n_pv, n_cart, n_fav
Target: n_buy

Empirical standard deviations:
  std(n_pv)  = 7.493511233952568
  std(n_cart)= 0.7232903347905415
  std(n_fav) = 0.5125972960494157

Empirical correlation matrix:
[[1.         0.43484194 0.24623258]
 [0.43484194 1.         0.04548658]
 [0.24623258 0.04548658 1.        ]]

Running PCA on X (no standardization, so variance dominates)...
Explained variance ratio:
[0.98822831 0.00748963 0.00428206]

Principal component vectors (rows):
  PC1: [0.99896483 0.04224007 0.01688314]
  PC2: [-0.03974873  0.99103319 -0.1275667 ]
  PC3: [-0.02212018  0.12676356  0.99168629]

Now running PCR with different numbers of PCs on Taobao...

Running PCR with k = 1 principal components...
  Train R^2 : 0.03860759265086877
  Test  R^2 : 0.029255209744913024
  Train RMSE: 0.3357864335599453
  Test  RMSE: 0.3039436660101213
  PCA components used by PCR:
    PC1: [0.99901962 0.04104473 0.01658671]
  Regression coefficients in PC space:
    [0.0089302

# PCA and PCR where one column varies less but is still important

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

print("UCL features: Quantity, UnitPrice, hour")
print("Target: line_total (= Quantity * UnitPrice in the raw data)\n")

sample = ucl_df.sample(n=min(5000, len(ucl_df)), random_state=42)

X1 = sample["Quantity"].values.astype(float)
X2 = sample["UnitPrice"].values.astype(float)
X3 = sample["hour"].values.astype(float)

X = np.column_stack([X1, X2, X3])
y = sample["line_total"].values

print("Empirical standard deviations:")
print("  std(Quantity)  =", X1.std(ddof=1))
print("  std(UnitPrice) =", X2.std(ddof=1))
print("  std(hour)      =", X3.std(ddof=1))
print()

corr = np.corrcoef(X, rowvar=False)
print("Empirical correlation matrix:")
print(corr)
print()

print("Running PCA on X (no standardization, so variance dominates)...")
pca = PCA(n_components=3)
pca.fit(X)

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)
print()

print("Principal component vectors (rows):")
for i, vec in enumerate(pca.components_, start=1):
    print(f"  PC{i}: {vec}")
print()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

def run_pcr(k):
    print(f"Running PCR with k = {k} principal components...")
    model = Pipeline([
        ("pca", PCA(n_components=k)),
        ("lr", LinearRegression())
    ])
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

    print("  Train R^2 :", train_r2)
    print("  Test  R^2 :", test_r2)
    print("  Train RMSE:", train_rmse)
    print("  Test  RMSE:", test_rmse)

    pca_step = model.named_steps["pca"]
    lr_step = model.named_steps["lr"]

    print("  PCA components used by PCR:")
    for i, vec in enumerate(pca_step.components_, start=1):
        print(f"    PC{i}: {vec}")

    print("  Regression coefficients in PC space:")
    print("   ", lr_step.coef_)
    print("  Intercept:")
    print("   ", lr_step.intercept_)
    print()

    return model

print("Now running PCR with different numbers of PCs on UCL...")
print("Quantity and UnitPrice dominate variance; hour contributes less.\n")

model1 = run_pcr(1)
model2 = run_pcr(2)
model3 = run_pcr(3)

print("Running ordinary linear regression on the original columns for comparison...")
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_test_lr = lr.predict(X_test)
print("  Test R^2 :", r2_score(y_test, y_pred_test_lr))
print("  Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test_lr)))
print("  Coefficients on original columns:")
print("   ", lr.coef_)
print("  Intercept:")
print("   ", lr.intercept_)
print()

print("Done.")
print("Expected qualitative result:")
print("  - PCA: early PCs align with Quantity and UnitPrice (largest variance).")
print("  - PCR with only 1 PC may miss useful structure in the remaining features.")
print("  - PCR with all 3 PCs should match OLS performance.")


UCL features: Quantity, UnitPrice, hour
Target: line_total (= Quantity * UnitPrice in the raw data)

Empirical standard deviations:
  std(Quantity)  = 60.25796574833261
  std(UnitPrice) = 37.73889309655962
  std(hour)      = 2.3172710132804486

Empirical correlation matrix:
[[ 1.         -0.01119708 -0.00747715]
 [-0.01119708  1.          0.00956054]
 [-0.00747715  0.00956054  1.        ]]

Running PCA on X (no standardization, so variance dominates)...
Explained variance ratio:
[0.71756409 0.28137498 0.00106093]

Principal component vectors (rows):
  PC1: [ 9.99933414e-01 -1.15362076e-02 -2.90583609e-04]
  PC2: [1.15363749e-02 9.99933285e-01 5.80856457e-04]
  PC3: [ 2.83863342e-04 -5.84170061e-04  9.99999789e-01]

Now running PCR with different numbers of PCs on UCL...
Quantity and UnitPrice dominate variance; hour contributes less.

Running PCR with k = 1 principal components...
  Train R^2 : 0.1376896676741679
  Test  R^2 : 0.3136048868058854
  Train RMSE: 64.14527789219798
  Test  